# Обзор результатов


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

root = Path.cwd()
if not (root / "report_results").exists():
    for p in root.parents:
        if (p / "report_results").exists():
            root = p
            break

table_dir = root / "report_results" / "tables"
fig_dir = root / "report_results" / "figures"

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 55)

def read_table(name):
    return pd.read_csv(table_dir / name)

def show_img(name):
    p = fig_dir / name
    if p.exists():
        display(Image(filename=str(p)))

case_names = {
    "heat_alpha01": "Heat, alpha=0.1",
    "helmholtz_m12_long": "Helmholtz, m=12",
    "helmholtz_m12_rs": "Helmholtz, m=12, resampling",
    "helmholtz_m12_resample128": "Helmholtz, m=12, long L-BFGS",
    "helmholtz_m7_rs": "Helmholtz, m=7",
    "helmholtz_m8_rs": "Helmholtz, m=8",
    "helmholtz_m10_resample128": "Helmholtz, m=10",
    "helmholtz_m11_rs": "Helmholtz, m=11",
    "convection_beta30": "Convection, beta=30",
    "convection_beta50": "Convection, beta=50, diagnostic",
    "burgers_nu0p002": "Burgers, delta=0.002",
    "burgers_nu0p001": "Burgers, delta=0.001",
    "fp16_summary": "FP16 failure cases",
}

def add_names(df):
    df = df.copy()
    if "case_id" in df.columns:
        df["name"] = df["case_id"].map(case_names).fillna(df["case_id"])
    return df


## 1. Запуски

In [ ]:
runs = read_table("all_runs_normalized.csv")
overview = read_table("task_overview.csv")

summary = pd.DataFrame({
    "метрика": ["всего запусков", "валидных", "плохих или невалидных", "задачи", "dtype"],
    "значение": [
        len(runs),
        int(runs["is_valid"].sum()),
        int(runs["is_bad"].sum()),
        ", ".join(sorted(runs["task_name"].dropna().unique())),
        ", ".join(sorted(runs["dtype"].dropna().unique())),
    ],
})
summary


In [ ]:
cols = [
    "task_name", "main_parameter_name", "main_parameter_value", "dtype",
    "n_total", "n_valid", "n_bad", "median_best_l2", "bad_rate",
]
overview[cols].sort_values(["task_name", "main_parameter_value", "dtype"]).head(20)


В логах есть много старых пробных запусков, поэтому плохих запусков довольно много. Я их не удаляю: они важны, чтобы не делать вывод по одному удачному seed.


## 2. Heat как простая проверка

Heat оставлен как простая задача. Здесь не ожидается драматической разницы между FP32 и FP64.


In [ ]:
main_cases = read_table("report_main_cases.csv")
heat = main_cases[main_cases["task"] == "heat1d"]
heat_view = add_names(heat)
heat_view[[
    "name", "parameter", "n_seed_fp32", "n_seed_fp64",
    "fp32_median_best_l2", "fp64_median_best_l2", "ratio",
]]


Heat нужен скорее как простая проверка: обе точности дают нормальный результат, поэтому дальше можно смотреть на более сложные PDE.


## 3. Helmholtz - основной блок

Главная часть отчёта строится вокруг Helmholtz. Здесь есть несколько сопоставимых запусков, где FP64 даёт меньшую медианную ошибку, и это не держится на одном seed.


In [ ]:
helm = read_table("report_helmholtz_cases.csv")
helm_view = add_names(helm)
helm_view[[
    "name", "parameter", "n_seed_fp32", "n_seed_fp64",
    "fp32_median_best_l2", "fp64_median_best_l2", "ratio",
    "bad_rate_fp32", "bad_rate_fp64",
]]


In [ ]:
show_img("report_helmholtz_ratio.png")
show_img("report_helmholtz_rs_sweep.png")
show_img("report_helmholtz_m12_curves.png")


In [ ]:
comp = read_table("fp32_fp64_comparison.csv")
helm_all = comp[comp["task_name"] == "helmholtz1d"].copy()
helm_all = helm_all[
    helm_all["main_parameter_value"].isin([5, 7, 8, 10, 11, 12, 15, 20, 30, 40])
]
cols = [
    "main_parameter_value", "fp32_n_valid", "fp64_n_valid",
    "fp32_median_best_l2", "fp64_median_best_l2", "fp64_over_fp32_median",
    "fp32_bad_rate", "fp64_bad_rate"
]
helm_all[cols].sort_values(["main_parameter_value"]).head(15)


Для основного текста лучше брать чистые Helmholtz-строки из `report_helmholtz_cases.csv`. Старый `helmholtz_main m=8` и очень большие `m` полезнее обсуждать как нестабильные режимы, а не как главное доказательство.


## 4. Burgers

Burgers получился смешанным. Здесь не видно чистой истории, что FP64 всегда лучше FP32.


In [ ]:
burg = comp[comp["task_name"] == "burgers1d"].copy()
cols = [
    "main_parameter_value", "fp32_n_valid", "fp64_n_valid",
    "fp32_median_best_l2", "fp64_median_best_l2", "fp64_over_fp32_median",
    "fp32_bad_rate", "fp64_bad_rate"
]
burg[cols].sort_values(["main_parameter_value"]).head(15)


In [ ]:
show_img("report_burgers_summary.png")


## 5. Convection

In [ ]:
conv = comp[comp["task_name"] == "convection1d"].copy()
conv_main = conv[conv["main_parameter_value"].eq(30)].copy()
cols = [
    "main_parameter_value", "fp32_n_valid", "fp64_n_valid",
    "fp32_median_best_l2", "fp64_median_best_l2", "fp64_over_fp32_median",
    "fp32_bad_rate", "fp64_bad_rate"
]
conv_main[cols].sort_values(["main_parameter_value"]).head(10)


In [ ]:
diag = read_table("report_diagnostic_cases.csv")
diag_view = add_names(diag)
diag_view[[
    "name", "parameter", "n_seed_fp32", "n_seed_fp64",
    "fp32_median_best_l2", "fp64_median_best_l2", "ratio",
]]


In [ ]:
show_img("report_convection_beta30_curves.png")
show_img("report_diagnostic_seed_sensitive.png")


## 6. FP16

В этих запусках он чаще показывает нестабильность.


In [ ]:
fp16 = read_table("fp16_summary.csv")
reason_col = fp16.columns[-1]
fp16_view = fp16[[
    "task_name", "main_parameter_name", "main_parameter_value",
    "n_total", "n_valid", "n_bad", "bad_rate"
]]
fp16_view


In [ ]:
show_img("report_fp16_summary.png")


## 7. Итоговые результаты


In [ ]:
main_view = add_names(main_cases)
main_view[[
    "name", "task", "parameter", "n_seed_fp32", "n_seed_fp64",
    "fp32_median_best_l2", "fp64_median_best_l2", "ratio",
    "bad_rate_fp32", "bad_rate_fp64", "label",
]]


In [ ]:
show_img("report_main_best_l2_by_dtype.png")
show_img("report_main_fp64_fp32_ratio.png")
show_img("report_main_seed_scatter.png")
